## importations

In [28]:
import sys, matplotlib
# print("Python path:", sys.executable)
# print("Matplotlib backend:", matplotlib.get_backend())




In [29]:
import matplotlib
matplotlib.use("QtAgg")   # ou "TkAgg" si QtAgg ne marche pas
import matplotlib.pyplot as plt


In [30]:
import matplotlib
matplotlib.get_backend()

# doit répondre 'QtAgg' et pas 'inlite' pour avoir des animations

'QtAgg'

In [31]:
"""

A* grid planning

author: Atsushi Sakai(@Atsushi_twi)
        Nikos Kanargias (nkana@tee.gr)

See Wikipedia article (https://en.wikipedia.org/wiki/A*_search_algorithm)

"""

import math

import matplotlib.pyplot as plt

show_animation = True


## constantes et variables

In [32]:
# table: 
Tx=300
Ty=200
#grenier:
Gr=[[60,155],[240,200]]
#garde mangers:
GM1=[[0,90],[20,70]]
GM2=[[60,0],[80,20]]
GM3=[[70,70],[90,90]]
GM4=[[115,135],[135,155]]
GM5=[[140,0],[160,20]]
GM6=[[140,70],[160,90]]
GM7=[[165,135],[185,155]]
GM8=[[210,70],[230,90]]
GM9=[[220,0],[240,20]]
GM10=[[280,70],[300,90]]


# start and goal
sx0,sy0 = 30,180
gx0,gy0 = 250,51

#cmd gardes manger:
cmd_gm1=0
cmd_gm2=0
cmd_gm3=0
cmd_gm4=0
cmd_gm5=0
cmd_gm6=0
cmd_gm7=0
cmd_gm8=0
cmd_gm9=0
cmd_gm10=0


## fonctions

### Astar

In [33]:


class AStarPlanner:

    def __init__(self, ox, oy, resolution, rr):
        """
        Initialize grid map for a star planning

        ox: x position list of Obstacles [m]
        oy: y position list of Obstacles [m]
        resolution: grid resolution [m]
        rr: robot radius[m]
        """

        self.resolution = resolution
        self.rr = rr
        self.min_x, self.min_y = 0, 0
        self.max_x, self.max_y = 0, 0
        self.obstacle_map = None
        self.x_width, self.y_width = 0, 0
        self.motion = self.get_motion_model()
        self.calc_obstacle_map(ox, oy)

    class Node:
        def __init__(self, x, y, cost, parent_index):
            self.x = x  # index of grid
            self.y = y  # index of grid
            self.cost = cost
            self.parent_index = parent_index

        def __str__(self):
            return str(self.x) + "," + str(self.y) + "," + str(
                self.cost) + "," + str(self.parent_index)

    def planning(self, sx, sy, gx, gy):
        """
        A star path search

        input:
            s_x: start x position [m]
            s_y: start y position [m]
            gx: goal x position [m]
            gy: goal y position [m]

        output:
            rx: x position list of the final path
            ry: y position list of the final path
        """

        start_node = self.Node(self.calc_xy_index(sx, self.min_x),
                               self.calc_xy_index(sy, self.min_y), 0.0, -1)
        goal_node = self.Node(self.calc_xy_index(gx, self.min_x),
                              self.calc_xy_index(gy, self.min_y), 0.0, -1)

        open_set, closed_set = dict(), dict()
        open_set[self.calc_grid_index(start_node)] = start_node

        while True:
            if len(open_set) == 0:
                print("Open set is empty..")
                break

            c_id = min(
                open_set,
                key=lambda o: open_set[o].cost + self.calc_heuristic(goal_node,
                                                                     open_set[
                                                                         o]))
            current = open_set[c_id]

            # show graph
            if show_animation:  # pragma: no cover
                plt.plot(self.calc_grid_position(current.x, self.min_x),
                         self.calc_grid_position(current.y, self.min_y), "xc")
                # for stopping simulation with the esc key.
                plt.gcf().canvas.mpl_connect('key_release_event',
                                             lambda event: [exit(
                                                 0) if event.key == 'escape' else None])
                if len(closed_set.keys()) % 10 == 0:
                    plt.pause(0.001)

            if current.x == goal_node.x and current.y == goal_node.y:
                print("Find goal")
                goal_node.parent_index = current.parent_index
                goal_node.cost = current.cost
                break

            # Remove the item from the open set
            del open_set[c_id]

            # Add it to the closed set
            closed_set[c_id] = current

            # expand_grid search grid based on motion model
            for i, _ in enumerate(self.motion):
                node = self.Node(current.x + self.motion[i][0],
                                 current.y + self.motion[i][1],
                                 current.cost + self.motion[i][2], c_id)
                n_id = self.calc_grid_index(node)

                # If the node is not safe, do nothing
                if not self.verify_node(node):
                    continue

                if n_id in closed_set:
                    continue

                if n_id not in open_set:
                    open_set[n_id] = node  # discovered a new node
                else:
                    if open_set[n_id].cost > node.cost:
                        # This path is the best until now. record it
                        open_set[n_id] = node

        rx, ry = self.calc_final_path(goal_node, closed_set)

        return rx, ry

    def calc_final_path(self, goal_node, closed_set):
        # generate final course
        rx, ry = [self.calc_grid_position(goal_node.x, self.min_x)], [
            self.calc_grid_position(goal_node.y, self.min_y)]
        parent_index = goal_node.parent_index
        while parent_index != -1:
            n = closed_set[parent_index]
            rx.append(self.calc_grid_position(n.x, self.min_x))
            ry.append(self.calc_grid_position(n.y, self.min_y))
            parent_index = n.parent_index

        return rx, ry

    @staticmethod
    def calc_heuristic(n1, n2):
        w = 1.0  # weight of heuristic
        d = w * math.hypot(n1.x - n2.x, n1.y - n2.y)
        return d

    def calc_grid_position(self, index, min_position):
        """
        calc grid position

        :param index:
        :param min_position:
        :return:
        """
        pos = index * self.resolution + min_position
        return pos

    def calc_xy_index(self, position, min_pos):
        return round((position - min_pos) / self.resolution)

    def calc_grid_index(self, node):
        return (node.y - self.min_y) * self.x_width + (node.x - self.min_x)

    def verify_node(self, node):
        px = self.calc_grid_position(node.x, self.min_x)
        py = self.calc_grid_position(node.y, self.min_y)

        if px < self.min_x:
            return False
        elif py < self.min_y:
            return False
        elif px >= self.max_x:
            return False
        elif py >= self.max_y:
            return False

        # collision check
        if self.obstacle_map[node.x][node.y]:
            return False

        return True

    def calc_obstacle_map(self, ox, oy):

        self.min_x = round(min(ox))
        self.min_y = round(min(oy))
        self.max_x = round(max(ox))
        self.max_y = round(max(oy))
        print("min_x:", self.min_x)
        print("min_y:", self.min_y)
        print("max_x:", self.max_x)
        print("max_y:", self.max_y)

        self.x_width = round((self.max_x - self.min_x) / self.resolution)
        self.y_width = round((self.max_y - self.min_y) / self.resolution)
        print("x_width:", self.x_width)
        print("y_width:", self.y_width)

        # obstacle map generation
        self.obstacle_map = [[False for _ in range(self.y_width)]
                             for _ in range(self.x_width)]
        for ix in range(self.x_width):
            x = self.calc_grid_position(ix, self.min_x)
            for iy in range(self.y_width):
                y = self.calc_grid_position(iy, self.min_y)
                for iox, ioy in zip(ox, oy):
                    d = math.hypot(iox - x, ioy - y)
                    if d <= self.rr:
                        self.obstacle_map[ix][iy] = True
                        break

    @staticmethod
    def get_motion_model():
        # dx, dy, cost
        motion = [[1, 0, 1],
                  [0, 1, 1],
                  [-1, 0, 1],
                  [0, -1, 1],
                  [-1, -1, math.sqrt(2)],
                  [-1, 1, math.sqrt(2)],
                  [1, -1, math.sqrt(2)],
                  [1, 1, math.sqrt(2)]]

        return motion


### main

In [34]:

def main():
    #print(__file__ + " start!!")

    # start and goal position
    sx = sx0 # [m]
    sy = sy0  # [m]
    gx = gx0  # [m]
    gy = gy0  # [m]
    grid_size = 2.0  # [m]
    robot_radius = 1.0  # [m]

    # set obstacle positions

        #/!\ il faut que le pt départ et arrivé soient plus ou moins enfermés pour que l'algo marche

    ox, oy = [], []

    def obs_rect(x1,y1='solo',x2='liste', y2='liste'):
        if y1=='solo':
            y1,x1=x1[1],x1[0]
        if x2=='liste':
            x2,y2=y1[0],y1[1]
            x1,y1=x1[0],x1[1]    
        if x1>x2:
            x1,x2=x2,x1
        if y1>y2:
            y1,y2=y2,y1            
        for i in range(x1,x2):
            ox.append(i)
            oy.append(y1)
            ox.append(i)
            oy.append(y2)

        for i in range(y1,y2):
            ox.append(x1)
            oy.append(i)
            ox.append(x2)
            oy.append(i)


    def obs_seg(x1, y1, x2='liste', y2='liste'):
        if x2=='liste':
            x2,y2=y1[0],y1[1]
            x1,y1=x1[0],x1[1]
        dx = abs(x2 - x1)
        dy = abs(y2 - y1)
        sx = 1 if x1 < x2 else -1
        sy = 1 if y1 < y2 else -1
        err = dx - dy
        x, y = x1, y1
        while True:
            ox.append(x)
            oy.append(y)
            if x == x2 and y == y2:
                break
            e2 = 2 * err
            if e2 > -dy:
                err -= dy
                x += sx
            if e2 < dx:
                err += dx
                y += sy


    def obs_poly(liste):    #liste de la forme: [[x1,y1],[x2,y2],...,[xn,yn]]
        print('flag1')
        nb_pt=len(liste)
        for i in range(nb_pt):
            if i<nb_pt-1:
                obs_seg(liste[i],liste[i+1])
            else:
                obs_seg(liste[i],liste[0])
                print('OOK')

    def adversaire(x0,y0,r):
        xp=x0+r
        xm=x0-r
        xpf=x0+0.75*r
        xmf=x0-0.75*r
        yp=y0+r
        ym=y0-r
        ypf=y0+0.75*r
        ymf=y0-0.75*r
        liste=[[xp,y0],[xmf,ypf],[x0,yp],[xpf,ypf],[xp,y0],[xpf,ymf],[x0,yp],[xmf,ymf]]
        return liste

        

    #table et grenier:
    obs_rect(0,0,Tx,Ty)
    obs_rect(Gr)

    #garde manger:
    l_cmd=[cmd_gm1,cmd_gm2,cmd_gm3,cmd_gm4,cmd_gm5,cmd_gm6,cmd_gm7,cmd_gm8,cmd_gm9,cmd_gm10]
    l_GM=[GM1,GM2,GM3,GM4,GM5,GM6,GM7,GM8,GM9,GM10]
    for i in range(len(l_cmd)):
        if l_cmd[i]==1:
            obs_rect(l_GM[i])

    #adversaire:
    # obs_poly(adversaire(70,70,5))


    #test
    # obs_poly([[10,10],[20,20],[10,20]])

    # obs_rect([[50,100],[100,150]])

    obs_seg(150, 150, 50, 50)
    p1,p2=[150, 150],[50, 51]
    # obs_seg(p1,p2)


    if show_animation:  # pragma: no cover
        plt.plot(ox, oy, ".k")
        plt.plot(sx, sy, "og")
        plt.plot(gx, gy, "xb")
        plt.grid(True)
        plt.axis("equal")

    a_star = AStarPlanner(ox, oy, grid_size, robot_radius)
    rx, ry = a_star.planning(sx, sy, gx, gy)

    if show_animation:  # pragma: no cover
        plt.plot(rx, ry, "-r")
        plt.pause(0.001)
        plt.show()

In [ ]:
    # dans costmap_update_node.pi:

    # def draw_enemy_robot(self, img, x, y):
    #     enemy_x = round(x / self.resolution)
    #     enemy_y = round(y / self.resolution)
    #     radius = round(np.ceil((self.enemy_robot_radius + self.robot_radius) / self.resolution))

    #     # Draw enemy robot
    #     cv2.circle(img, (enemy_x, enemy_y), radius, 100, -1)


# voir pour utiliser numpy pour faire la grille 
# liste à renvoyer: [[x1,y1],[]]

## executables

In [36]:
# main()

In [37]:

if __name__ == '__main__':
    main()


min_x: 0
min_y: 0
max_x: 300
max_y: 200
x_width: 150
y_width: 100


qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
qt.qpa.wayland: Wayland does not support QWindow

Find goal


qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
